<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page, identified by `content_id`, belonging to one client (`client_id`). This is a flat, single-table CSV (`content_refresh_anonymized.csv`) — there is no join across separate fact/dimension tables, and no explicit calendar date column anywhere in the data.

Instead of absolute dates, the "time window" here is relative, expressed as rolling day-counts as of an implicit snapshot moment: `impressions_90d` (last 90 days), `impressions_last_30d` vs `impressions_prev_30d` (the most recent 30-day window vs. the 30 days before that), `content_age_days` (days since the page was created), and `days_since_last_update` (days since the page was last edited). So every claim in this contract is scoped to "as of the snapshot this CSV was pulled," not to a specific calendar month.

In [2]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))
print("Unique content_id values:", df["content_id"].nunique())
print("Duplicate content_id rows (should be 0 if grain holds):", df["content_id"].duplicated().sum())
print("Unique client_id values:", df["client_id"].nunique())

date_like_cols = [c for c in df.columns if "date" in c.lower()]
print("Columns with 'date' in the name (expect none):", date_like_cols)

window_cols = [c for c in df.columns if "90d" in c or "30d" in c or "age_days" in c]
print("Relative time-window columns found:", window_cols)

Total rows: 30000
Unique content_id values: 30000
Duplicate content_id rows (should be 0 if grain holds): 0
Unique client_id values: 32
Columns with 'date' in the name (expect none): ['days_since_last_update']
Relative time-window columns found: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (used to train/rank): `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `days_since_last_update`, `content_age_days`, `word_count`, `freshness_tier`, `days_with_impressions`.

**Label**: `trend_direction` (specifically, whether it equals `"down"`), which is an observed later outcome, not a hand-written rule. `trend_pct` is the continuous version of the same signal.

**Context** (useful for understanding results, not fed to the model as predictive signal): `content_type`, `main_intent`, `provider_used`, `model_used`, `competition_level`.

**Excluded, with why**:
- `client_id`, `content_id` — these are row identifiers, not predictive signals, and are excluded from any printed output to avoid exposing per-client data (self-check requirement: no client names/private data in outputs).
- `impressions_last_30d`, `impressions_prev_30d` — these are excluded from the **feature set** because they can be used to directly derive a decline label (`impressions_last_30d < impressions_prev_30d`), which would leak the label into the features. They are verified as a leakage risk in Section 3.

In [3]:

features = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "days_since_last_update", "content_age_days",
            "word_count", "freshness_tier", "days_with_impressions"]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_type", "main_intent", "provider_used", "model_used", "competition_level"]
excluded_cols = ["client_id", "content_id", "impressions_last_30d", "impressions_prev_30d"]

all_named = set(features + label_cols + context_cols + excluded_cols)
missing_from_contract = [c for c in df.columns if c not in all_named]
print("Columns not yet sorted into a bucket (review before final submission):")
print(missing_from_contract)

print("\nFeature summary stats:")
print(df[features].describe().T[["mean", "min", "max"]])

Columns not yet sorted into a bucket (review before final submission):
['search_volume', 'competition', 'cpc', 'char_count', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_sessions', 'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'age_tier', 'age_tier_order', 'word_count_tier', 'char_count_tier', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Feature summary stats:
                               mean   min       max
impressions_90d         5200.366300   1.0  517715.0
ctr                        0.510733   0.0     100.0
avg_position              16.342380   0.0     245.0
engagement_rate            2.534520   0.0     100.0
scroll_rate               18.212921   0.0     300.0
days_since_last_update    46.098300   1.0     373.0
content_age_days         256.167800  90.0     564.0
word_count              3107.760325   8.0    9546.0
days_with_impressions     61.454033

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three things need verifying: (1) missing data in the features we plan to use, (2) whether the rolling windows are internally consistent, and (3) whether the excluded columns really would leak the label if we kept them in — tested directly with an honest-vs-leaky model comparison rather than just asserted.

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
features = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "days_since_last_update", "content_age_days",
            "word_count", "freshness_tier", "days_with_impressions"]

# --- Query 1: missing values in planned features ---
print("=== Query 1: Missing values in feature set ===")
print(df[features].isnull().sum())

# --- Query 2: window consistency check ---
print("\n=== Query 2: Window consistency ===")
overlap_ok = (df["impressions_last_30d"] + df["impressions_prev_30d"] <= df["impressions_90d"]).mean()
print(f"Share of rows where last_30d + prev_30d <= impressions_90d: {overlap_ok:.1%}")

print("\n=== Query 3: Leakage check ===")
work = df.copy()
work["avg_position"] = work["avg_position"].fillna(work["avg_position"].median())
y = (work["trend_direction"] == "down").astype(int)

numeric_features = [c for c in features if pd.api.types.is_numeric_dtype(work[c])]
X_honest_raw = work[numeric_features].fillna(0)
X_honest = StandardScaler().fit_transform(X_honest_raw)
clf_honest = LogisticRegression(max_iter=2000, random_state=42)
clf_honest.fit(X_honest, y)
auc_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"Honest model ROC-AUC (excluded cols left out): {auc_honest:.4f}")

X_leaky_raw = X_honest_raw.copy()
X_leaky_raw["impressions_last_30d"] = work["impressions_last_30d"]
X_leaky = StandardScaler().fit_transform(X_leaky_raw)
clf_leaky = LogisticRegression(max_iter=2000, random_state=42)
clf_leaky.fit(X_leaky, y)
auc_leaky = roc_auc_score(y, clf_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky model ROC-AUC (impressions_last_30d added back in): {auc_leaky:.4f}")
print(f"AUC jump from adding the excluded column: {auc_leaky - auc_honest:+.4f}")
print("A large jump here confirms impressions_last_30d is correctly excluded as a leakage risk.")

=== Query 1: Missing values in feature set ===
impressions_90d              0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
days_since_last_update       0
content_age_days             0
word_count                7699
freshness_tier               0
days_with_impressions        0
dtype: int64

=== Query 2: Window consistency ===
Share of rows where last_30d + prev_30d <= impressions_90d: 100.0%

=== Query 3: Leakage check ===
Honest model ROC-AUC (excluded cols left out): 0.6688
Leaky model ROC-AUC (impressions_last_30d added back in): 0.8234
AUC jump from adding the excluded column: +0.1546
A large jump here confirms impressions_last_30d is correctly excluded as a leakage risk.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Based only on what Section 3's queries actually show (not on assumptions carried over from a different dataset):

- **No calendar dates.** There is no date column anywhere in this file (confirmed in Section 1). We can say a page's trend is "down as of the snapshot," but not *when* the decline happened or how recent it is in absolute time.
- **No causal information.** `trend_direction` is an observed outcome, not evidence that any specific factor caused the decline. Any ranking this produces is directional/decision-support, not a causal claim.
- **No refresh-history flag.** Nothing in the data indicates whether a page was already manually refreshed during the observed window — this is a real confound for any future claim like "refreshing improves ranking," since we can't currently separate "never touched and declining" from "recently refreshed and still catching up."
- **Missing values, if any, in `avg_position`** should be treated as "page had no measurable average position in the window" rather than imputed silently as zero — Section 3, Query 1 reports the actual count so this can be handled deliberately rather than guessed at.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.